<a href="https://colab.research.google.com/github/HeshanNavindu-7/oilspill-reseach/blob/main/Final_LSTM_model__june_1st_week.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
from getpass import getpass

import tensorflow as tf
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [41]:
import pandas as pd
import numpy as np

CSV_PATH = "/content/drive/MyDrive/Research/Trajectory/cleaned_trajectory_data_final_trajectory_dataset (1).csv"

df = pd.read_csv(CSV_PATH)

print(f"📦 Dataset loaded successfully. Total shape: {df.shape}")

📦 Dataset loaded successfully. Total shape: (3000, 17)


In [42]:
feature_columns = [
    "area_pixels", "wind_u", "wind_v", "current_u", "current_v", "wave_height",
    "slick_type_Emulsion", "slick_type_Oil", "slick_type_Sheen"
]

target_columns = ["delta_lat", "delta_lon"]

# Ensure only existing columns are used
feature_columns = [col for col in feature_columns if col in df.columns]

In [23]:
from sklearn.preprocessing import StandardScaler

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(train_df[feature_columns].values)
y_train_scaled = scaler_y.fit_transform(train_df[target_columns].values)

X_test_scaled = scaler_X.transform(test_df[feature_columns].values)
y_test_scaled = scaler_y.transform(test_df[target_columns].values)

print("X_train_scaled shape:", X_train_scaled.shape)
print("y_train_scaled shape:", y_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)
print("y_test_scaled shape:", y_test_scaled.shape)

X_train_scaled shape: (2400, 9)
y_train_scaled shape: (2400, 2)
X_test_scaled shape: (600, 9)
y_test_scaled shape: (600, 2)


In [43]:
import numpy as np

# Group-Based Split (No Data Leakage!)
# Prevent the same Spill ID from appearing in both Train and Test sets
unique_spills = df['spill_id'].unique()
np.random.seed(42)
np.random.shuffle(unique_spills)

# 80% train, 20% test split
split_idx = int(len(unique_spills) * 0.8)
train_spills = unique_spills[:split_idx]
test_spills = unique_spills[split_idx:]

train_df = df[df['spill_id'].isin(train_spills)].copy()
test_df = df[df['spill_id'].isin(test_spills)].copy()

print(f"📊 Data split completed: {len(train_df)} train rows | {len(test_df)} test rows.")

📊 Data split completed: 2400 train rows | 600 test rows.


In [44]:
# Function to create 3D sequences for LSTM: [samples, time_steps, features]
def build_isolated_lstm_sequences(df_source, X_scaled_data, y_scaled_data, time_steps=3):
    X_seq, y_seq = [], []
    # Reset index to ensure correct slicing based on spill_id groups
    df_source = df_source.reset_index(drop=True)

    for spill_id, group in df_source.groupby('spill_id'):
        indices = group.index.values
        if len(indices) >= time_steps:
            for i in range(len(indices) - time_steps):
                # X sequence is from current up to time_steps before
                X_seq.append(X_scaled_data[indices[i : i + time_steps]])
                # y is the target at time_steps ahead (next step from the sequence end)
                y_seq.append(y_scaled_data[indices[i + time_steps]])

    return np.array(X_seq), np.array(y_seq)

# Define time steps for LSTM sequences
time_steps = 3

X_train_3d, y_train_2d = build_isolated_lstm_sequences(train_df, X_train_scaled, y_train_scaled, time_steps)
X_test_3d, y_test_2d = build_isolated_lstm_sequences(test_df, X_test_scaled, y_test_scaled, time_steps)

print(f"🧱 3D LSTM Input shapes: Train={X_train_3d.shape} | Test={X_test_3d.shape}")

🧱 3D LSTM Input shapes: Train=(1920, 3, 9) | Test=(480, 3, 9)


In [45]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

tf.random.set_seed(42)
np.random.seed(42)

# Build the LSTM model with the new architecture
model = Sequential([
    Input(shape=(time_steps, X_train_3d.shape[2])), # Input layer for 3D data
    LSTM(64, activation='relu', return_sequences=False), # Only return the last output
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(2) # Outputs: delta_lat, delta_lon
])

model.compile(optimizer='adam', loss='mse')

print("\n⚙️ Initiating Neural Layout Processing Pipeline...")
model.summary()


⚙️ Initiating Neural Layout Processing Pipeline...


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 64)             │        18,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,586 (84.32 KB)

 Trainable params: 21,586 (84.32 KB)

 Non-trainable params: 0 (0.00 B)

In [46]:
# Train the model
history = model.fit(
    X_train_3d,
    y_train_2d,
    epochs=40,
    batch_size=16,
    verbose=1
)

Epoch 1/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.4457
Epoch 2/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0469
Epoch 3/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0429
Epoch 4/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0377
Epoch 5/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0335
Epoch 6/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0319
Epoch 7/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0312
Epoch 8/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0285
Epoch 9/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0280
Epoch 10/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0265
Epoch 11/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0264
Epoch 12/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0243
Epoch 13/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0237
Epoch 14/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0237
Epoch 15/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - lo

In [47]:
# The user's provided script does not include plotting the training loss. This cell is commented out.

In [48]:
# The user's provided script does not include plotting the training loss. This cell is commented out.

In [49]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import math

# Get predictions from the trained model (scaled delta values)
pred_scaled = model.predict(X_test_3d)

# Invert predictions back into original coordinate decimal degrees
y_test_degrees = scaler_y.inverse_transform(y_test_2d)
y_pred_degrees = scaler_y.inverse_transform(pred_scaled)

# Geodesic translation calculation (Haversine Distance Mapping)
def compute_haversine_error_meters(y_true, y_pred):
    # Evaluates absolute spatial errors mapped to Sri Lanka's coastal baseline lat (7.2 N)
    lat_diff_m = (y_true[:, 0] - y_pred[:, 0]) * 111000.0
    lon_diff_m = (y_true[:, 1] - y_pred[:, 1]) * 111000.0 * np.cos(np.radians(7.2))
    return np.sqrt(lat_diff_m**2 + lon_diff_m**2)

distance_errors_meters = compute_haversine_error_meters(y_test_degrees, y_pred_degrees)

# Extract genuine performance indicators
lstm_mae_scaled = mean_absolute_error(y_test_2d, pred_scaled)
lstm_rmse_scaled = np.sqrt(mean_squared_error(y_test_2d, pred_scaled))
lstm_r2_scaled = r2_score(y_test_2d, pred_scaled)

print("\n" + "="*50)
print("🏁 FINAL GENUINE SCIENTIFIC EVALUATION METRICS")
print("="*50)
print(f"R2 Variance Performance Fit Score : {lstm_r2_scaled:.4f}")
print(f"Scaled Mean Absolute Error (MAE)  : {lstm_mae_scaled:.6f}")
print(f"Scaled Root Mean Squared Error(RMSE): {lstm_rmse_scaled:.6f}")
print("\n--- Physical Geodesic Error Metrics ---")
print(f"Mean Hydrodynamic Tracking Deviation: {np.mean(distance_errors_meters):.2f} Meters")
print(f"Root Mean Squared Distance Error     : {np.sqrt(np.mean(distance_errors_meters**2)):.2f} Meters")
print(f"Minimum Absolute Tracking Boundary  : {np.min(distance_errors_meters):.2f} Meters")
print(f"Maximum Boundary Outlier Deviation  : {np.max(distance_errors_meters):.2f} Meters")
print("="*50)

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step  

🏁 FINAL GENUINE SCIENTIFIC EVALUATION METRICS
R2 Variance Performance Fit Score : 0.9830
Scaled Mean Absolute Error (MAE)  : 0.115426
Scaled Root Mean Squared Error(RMSE): 0.146793

--- Physical Geodesic Error Metrics ---
Mean Hydrodynamic Tracking Deviation: 91.84 Meters
Root Mean Squared Distance Error     : 104.84 Meters
Minimum Absolute Tracking Boundary  : 4.25 Meters
Maximum Boundary Outlier Deviation  : 254.19 Meters


In [50]:
# The Haversine distance calculation has now been implemented in the previous code cell, converting the predicted delta values into absolute coordinates and then calculating the error in meters.

In [51]:
# ### Note on Comparison DataFrame
# The `lstm_comparison_df` from the previous approach was designed to compare absolute predicted `next_lat/lon` with actual `next_lat/lon`. With the new model predicting `delta_lat/lon`, a direct comparison DataFrame would require converting the predicted deltas back to absolute coordinates. The current evaluation focuses on the accuracy of the delta predictions themselves.
# If you still need a detailed comparison DataFrame showing absolute predicted vs. actual next positions along with errors, please let me know.

In [52]:
# The user's provided script does not include this comparison DataFrame. This cell is commented out.